In [1]:
import rasterio
from rasterio.windows import Window
from rasterio.env import Env 
import numpy as np

tile_size = 2048
red_path = r'C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B04_10m.jp2'
nir_path = r'C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B08_10m.jp2'
green_path = r'C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B03_10m.jp2'
blue_path = r'C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B02_10m.jp2'
output_path = r'Stacked_bands.tif'

with Env(GDAL_CACHEMAX=256):
    with rasterio.open(red_path) as red_src, \
         rasterio.open(nir_path) as nir_src, \
         rasterio.open(green_path) as green_src, \
         rasterio.open(blue_path) as blue_src:
        
        profile = red_src.profile.copy()
        profile.update(
            count=4,
            dtype="float32",
            driver="GTiff",
            width=red_src.width,
            height=red_src.height,
            transform=red_src.transform,
            compress='lzw',
            tiled=True,
            blockxsize=256,
            blockysize=256
        )

        with rasterio.open(output_path, "w", **profile) as dst:
            for row in range(0, red_src.height, tile_size):
                for col in range(0, red_src.width, tile_size):

                    
                    win_h = min(tile_size, red_src.height - row)
                    win_w = min(tile_size, red_src.width - col)
                    window = Window(col, row, win_w, win_h)
                    
                    stack = np.stack([
                        red_src.read(1, window=window),
                        nir_src.read(1, window=window),
                        green_src.read(1, window=window),
                        blue_src.read(1, window=window)
                    ]).astype("float32") / 10000.0

                    if stack.max()==0:
                        continue
                    dst.write(stack, window=window)

print("Stacking complete. Output saved to", output_path)

Stacking complete. Output saved to Stacked_bands.tif
